## 1. Setup

### Libraries

In [1]:
# weather data 
import meteostat as ms

# from src
from src.spark_config import initialize_spark # spark session
from src.helper_functions import get_coordinates, get_paths 

# spark biblioteker fra labene
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, expr, isnan, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, input_file_name,
    regexp_extract, min, max, year
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

# other libraries
import pandas as pd
import numpy as np
import regex as re
from datetime import datetime, timedelta

# specify java version
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

### Initialize

In [2]:
# set spark session
spark = initialize_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/27 20:19:05 WARN Utils: Your hostname, LaptopEHM, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/27 20:19:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/27 20:19:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.3.0
Application ID: local-1795807149370
Spark UI: http://10.255.255.254:4040


### Define Data Paths

In [3]:
# paths for statsbomb data
paths = get_paths()

## 2. Matches

### Load and Filter La Liga Matches

In [4]:
matches_df = spark.read.parquet("data/bronze/statsbomb/matches")

# to make sure we dont have duplicated matches due to multiple ingestions
latest_ingest_date = matches_df.agg(
    spark_max("ingest_date")
).first()[0]

matches_df = matches_df.filter(
    col("ingest_date") == latest_ingest_date
)
laliga_matches = matches_df.filter(
    col("competition.competition_name") == "La Liga")

### Validation Rules

In [5]:
invalid_match = (
    col("match_id").isNull() |
    col("kick_off").isNull() |
    col("stadium.name").isNull() |
    col("season.season_name").isNull() |
    to_date(col("match_date")).isNull() |
    col("home_score").isNull() |
    col("away_score").isNull() |
    (col("home_score") < 0) |
    (col("away_score") < 0)
)

duplicate_match_ids = laliga_matches.groupBy("match_id").count().filter(col("count") > 1).select("match_id")

### Data Quality Before Cleaning

In [6]:
# data quality of matches before cleaning
print(f"Number of matches in La Liga (before cleaning): ", laliga_matches.count())
print(f"Number of unique matches in La Liga (before cleaning): ", laliga_matches.select("match_id").distinct().count())
print(f"Duplicate match ids (before cleaning): ", duplicate_match_ids.count())
print(f"Invalid matches (before cleaning): ", laliga_matches.filter(invalid_match).count())

Number of matches in La Liga (before cleaning):  867
Number of unique matches in La Liga (before cleaning):  867
Duplicate match ids (before cleaning):  0
Invalid matches (before cleaning):  0


### Validation and Quarantine

In [7]:
quarantine_matches = laliga_matches.filter(invalid_match)

valid_matches = laliga_matches.filter(~invalid_match)

duplicate_matches = valid_matches.join(
    duplicate_match_ids, on="match_id", how="inner"
)

quarantine_matches = quarantine_matches.unionByName(duplicate_matches)

valid_matches = valid_matches.join(
    duplicate_match_ids, on="match_id", how="left_anti"
)

print(f"Quarantined matches: ", quarantine_matches.count())
print(f"Valid matches: ", valid_matches.count())

Quarantined matches:  0
Valid matches:  867


### Transform Matches

In [8]:
# forslag til første utdrag av data
silver_matches = valid_matches.select(
        col("match_id"),
        col("kick_off"),
        col("stadium.name").alias("stadium"),
        col("home_team.home_team_name").alias("home_team"),
        col("away_team.away_team_name").alias("away_team"),
        col("home_team.home_team_id").alias("home_team_id"),
        col("away_team.away_team_id").alias("away_team_id"),
        col("home_score"), # sånn at vi kan se på om hjemmelaget spiller bedre i ulike værforhold?
        col("away_score"),
        col("season.season_name").alias("season"),
        to_date("match_date").alias("match_date")
        )

silver_matches.show(5)
silver_matches.printSchema()

+--------+------------+--------------------+--------------------+--------------+------------+------------+----------+----------+---------+----------+
|match_id|    kick_off|             stadium|           home_team|     away_team|home_team_id|away_team_id|home_score|away_score|   season|match_date|
+--------+------------+--------------------+--------------------+--------------+------------+------------+----------+----------+---------+----------+
| 3825739|17:00:00.000|            Bernabéu|         Real Madrid|Sporting Gijón|         220|        1041|         5|         1|2015/2016|2016-01-17|
| 3825848|20:00:00.000|Estadio Ciudad de...|          Levante UD|         Eibar|         221|         322|         2|         2|2015/2016|2015-09-23|
| 3825895|22:00:00.000|Estadio de Gran C...|          Las Palmas|       Sevilla|         208|         213|         2|         0|2015/2016|2015-09-23|
| 3825894|18:15:00.000|Estadio Abanca-Ri...|RC Deportivo La C...|        Getafe|         219|       

### Data Quality After Cleaning

In [9]:
# data quality of matches after cleaning
print(f"Number of matches in La Liga (after cleaning):", silver_matches.count())
print(f"Number of unique matches in La Liga (after cleaning):", silver_matches.select("match_id").distinct().count())
print(f"Duplicate match ids (after cleaning):", silver_matches.groupBy("match_id").count().filter(col("count") > 1).count())
print(f"Invalid matches (after cleaning):", valid_matches.filter(invalid_match).count())
print(f"Quarantined matches:", quarantine_matches.count())

Number of matches in La Liga (after cleaning): 867
Number of unique matches in La Liga (after cleaning): 867
Duplicate match ids (after cleaning): 0
Invalid matches (after cleaning): 0
Quarantined matches: 0


In [10]:
matches_df.printSchema()

root
 |-- away_score: long (nullable = true)
 |-- away_team: struct (nullable = true)
 |    |-- away_team_gender: string (nullable = true)
 |    |-- away_team_group: string (nullable = true)
 |    |-- away_team_id: long (nullable = true)
 |    |-- away_team_name: string (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- managers: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- country: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- dob: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- nickname: string (nullable = true)
 |-- competition: struct (nullable = true)
 |    |-- competition_id: long (nullable = true)
 |    |-- competition_nam

## 3. Events

### Load and Filter La Liga Events

In [11]:
events_df = spark.read.parquet("data/bronze/statsbomb/events")

# to make sure we dont have duplicated events due to multiple ingestions
latest_ingest_date = events_df.agg(
    spark_max("ingest_date")
).first()[0]

events_df = events_df.filter(
    col("ingest_date") == latest_ingest_date
)

laliga_events = events_df.join(
    laliga_matches.select("match_id"),
    on="match_id",
    how="left_semi"
)

### Validation Rules

In [12]:
invalid_event = (
    col("id").isNull() |
    col("match_id").isNull() |
    col("type.name").isNull() |
    (
        (col("type.name") == "Pass") & col("pass").isNull()
    )
)

duplicate_event_ids = (
    laliga_events
    .groupBy("id")
    .count()
    .filter(col("count") > 1)
    .select("id")
)

### Data Quality Before Cleaning

In [13]:
print(f"Number of events in La Liga (before cleaning): ", laliga_events.count())
print(f"Number of unique events in La Liga (before cleaning): ", laliga_events.select("id").distinct().count())
print(f"Duplicate event ids (before cleaning): ", duplicate_event_ids.count())
print(f"Invalid events (before cleaning): ", laliga_events.filter(invalid_event).count())


Number of events in La Liga (before cleaning):  3132716


Number of unique events in La Liga (before cleaning):  3132716


Duplicate event ids (before cleaning):  0


Invalid events (before cleaning):  0


### Validation and Quarantine

In [14]:
quarantine_events = laliga_events.filter(invalid_event)

valid_events = laliga_events.filter(~invalid_event)

duplicate_events = valid_events.join(
    duplicate_event_ids,
    on="id",
    how="inner"
)

quarantine_events = quarantine_events.unionByName(duplicate_events)

valid_events = valid_events.join(
    duplicate_event_ids,
    on="id",
    how="left_anti"
)

# Events without a valid match
orphan_events = valid_events.join(
    silver_matches.select("match_id"),
    on="match_id",
    how="left_anti"
)

quarantine_events = quarantine_events.unionByName(orphan_events)

valid_events = valid_events.join(
    silver_matches.select("match_id"),
    on="match_id",
    how="left_semi"
)

print("Quarantined events:", quarantine_events.count())
print("Valid events:", valid_events.count())

Quarantined events: 0


Valid events: 3132716


### Transform Events

In [15]:
silver_events = valid_events.select(
    col("match_id"),
    col("id").alias("event_id"),
    col("type.name").alias("event_type"),
    col("pass"),
    col("team.name").alias("team"),
    col("player.name").alias("player"),
    col("shot"),
    col("foul_committed")
    )

In [16]:
silver_events.printSchema()
silver_events.show(5)

root
 |-- match_id: long (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- pass: struct (nullable = true)
 |    |-- aerial_won: boolean (nullable = true)
 |    |-- angle: double (nullable = true)
 |    |-- assisted_shot_id: string (nullable = true)
 |    |-- body_part: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- cross: boolean (nullable = true)
 |    |-- cut_back: boolean (nullable = true)
 |    |-- deflected: boolean (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- goal_assist: boolean (nullable = true)
 |    |-- height: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- inswinging: boolean (nullable = true)
 |    |-- length: double (nullable = true)
 |    |-- miscommunication: boolean (nullable = true)


+--------+--------------------+-------------+--------------------+-------------+--------------------+----+--------------+
|match_id|            event_id|   event_type|                pass|         team|              player|shot|foul_committed|
+--------+--------------------+-------------+--------------------+-------------+--------------------+----+--------------+
| 3773593|00001fec-12f6-483...|         Pass|{NULL, 0.8306634,...|    Barcelona|Sergi Roberto Car...|NULL|          NULL|
|  303725|00005603-35a4-484...|Ball Receipt*|                NULL|   Villarreal| Bruno Soriano Llido|NULL|          NULL|
| 3825578|00007386-b8ba-404...|         Pass|{NULL, 0.5116587,...|Athletic Club|     Aymeric Laporte|NULL|          NULL|
| 3773497|0000bbd5-df0a-419...|        Carry|                NULL|  Real Madrid|         Luka Modrić|NULL|          NULL|
| 3773526|0000bfa9-1284-4ad...|         Pass|{NULL, -0.1907555...|    Barcelona|  Samuel Yves Umtiti|NULL|          NULL|
+--------+--------------

### Data Quality After Cleaning

In [17]:
print(f"Number of events in La Liga (after cleaning):", silver_events.count())
print(f"Number of unique events in La Liga (after cleaning):", silver_events.select("event_id").distinct().count())
print(f"Duplicate event ids (after cleaning):", silver_events.groupBy("event_id").count().filter(col("count") > 1).count())
print(f"Invalid events (after cleaning):", valid_events.filter(invalid_event).count())
print(f"Quarantined events:", quarantine_events.count())

Number of events in La Liga (after cleaning): 3132716


Number of unique events in La Liga (after cleaning): 3132716


Duplicate event ids (after cleaning): 0


Invalid events (after cleaning): 0


Quarantined events: 0


In [18]:
events_df.printSchema()

root
 |-- 50_50: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_receipt: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_recovery: struct (nullable = true)
 |    |-- recovery_failure: boolean (nullable = true)
 |-- block: struct (nullable = true)
 |    |-- deflection: boolean (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |-- carry: struct (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- clearance: struct (nullable = true)
 |    |-- aerial_won: boolean (nullable = true)
 |    |-- body_part: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- head: boolean (nullable = true)
 |    |-- left_f

## 4. Weather 

### Load Weather Data

In [19]:
weather_df = spark.read.parquet("data/bronze/meteostat")

# to make sure we dont have duplicated events due to multiple ingestions
latest_ingest_date = weather_df.agg(
    spark_max("ingest_date")
).first()[0]

weather_df = weather_df.filter(col("ingest_date") == latest_ingest_date)

# Filter weather observations for La Liga matches
laliga_weather = weather_df.join(
    laliga_matches.select("match_id"),
    on="match_id",
    how="left_semi"
)

### Validation Rules

In [20]:
weather_columns = ["temp", "rhum", "prcp", "wspd"]

invalid_temp = ~col("temp").between(-50, 60) & ~isnan(col("temp"))
invalid_rhum = ~col("rhum").between(0, 100) & ~isnan(col("rhum"))
invalid_prcp = col("prcp") < 0
invalid_wspd = col("wspd") < 0

invalid_weather = (
    col("match_id").isNull() |
    col("time").isNull() |
    invalid_temp |
    invalid_rhum |
    invalid_prcp |
    invalid_wspd
)

invalid_weather = coalesce(invalid_weather, lit(False))

# Check for duplicate weather observations per match and timestamp
duplicate_weather = (laliga_weather.groupBy("match_id", "time").count().filter(col("count") > 1))

### Data Quality Before Cleaning

In [21]:
print(f"Number of La Liga weather observations (before cleaning):", laliga_weather.count())
print(f"Number of La Liga matches with weather (before cleaning):", laliga_weather.select("match_id").distinct().count())
print(f"Invalid weather observations (before cleaning):", laliga_weather.filter(invalid_weather).count())
print("Duplicate match_id and time combinations (before cleaning):", duplicate_weather.count())

# Missing values for each weather variable
for column in weather_columns:
    missing = laliga_weather.filter(col(column).isNull() | isnan(col(column))).count()

    print(f"Missing values in {column}: {missing}")

Number of La Liga weather observations (before cleaning): 6148
Number of La Liga matches with weather (before cleaning): 491
Invalid weather observations (before cleaning): 0
Duplicate match_id and time combinations (before cleaning): 0
Missing values in temp: 1
Missing values in rhum: 3
Missing values in prcp: 6097
Missing values in wspd: 13


### Validation and Quarantine

In [22]:
quarantine_weather = laliga_weather.filter(invalid_weather)
valid_weather = laliga_weather.filter(~invalid_weather)

# quarantine duplicate weather observations
duplicate_observations = valid_weather.join(
    duplicate_weather,
    on=["match_id", "time"],
    how="left_semi"
)

quarantine_weather = quarantine_weather.unionByName(duplicate_observations)

valid_weather = valid_weather.join(
    duplicate_weather,
    on=["match_id", "time"],
    how="left_anti"
)

# Quarantine weather observations without a valid match
orphan_weather = valid_weather.join(
    silver_matches.select("match_id"),
    on="match_id",
    how="left_anti"
)

quarantine_weather = quarantine_weather.unionByName(orphan_weather)

valid_weather = valid_weather.join(
    silver_matches.select("match_id"),
    on="match_id",
    how="left_semi"
)

print("Quarantined weather observations:", quarantine_weather.count())
print("Valid weather observations:", valid_weather.count())

Quarantined weather observations: 0
Valid weather observations: 6148


### Transform Weather Data

In [ ]:
# Convert NaN to NULL
clean_weather = valid_weather.replace(float("nan"), None, subset=weather_columns)

# combine match date and kick-off into a timestamp
match_times = silver_matches.select("match_id", to_timestamp(concat(col("match_date"), lit(" "), col("kick_off"))).alias("kickoff_time"))

# join weather observations with match times
weather_with_kickoff = clean_weather.join(
    match_times,
    on="match_id",
    how="inner"
)

weather_with_kickoff.select("match_id", "time", "kickoff_time").show(15, truncate=False)

+--------+-------------------+-------------------+
|match_id|time               |kickoff_time       |
+--------+-------------------+-------------------+
|70220   |2011-09-17 14:00:00|2011-09-17 20:00:00|
|70220   |2011-09-17 15:00:00|2011-09-17 20:00:00|
|70220   |2011-09-17 16:00:00|2011-09-17 20:00:00|
|70220   |2011-09-17 17:00:00|2011-09-17 20:00:00|
|70220   |2011-09-17 18:00:00|2011-09-17 20:00:00|
|70220   |2011-09-17 19:00:00|2011-09-17 20:00:00|
|70220   |2011-09-17 20:00:00|2011-09-17 20:00:00|
|70220   |2011-09-17 21:00:00|2011-09-17 20:00:00|
|70220   |2011-09-17 22:00:00|2011-09-17 20:00:00|
|70220   |2011-09-17 23:00:00|2011-09-17 20:00:00|
|70220   |2011-09-18 00:00:00|2011-09-17 20:00:00|
|70220   |2011-09-18 01:00:00|2011-09-17 20:00:00|
|70220   |2011-09-18 02:00:00|2011-09-17 20:00:00|
|69328   |2012-02-19 13:00:00|2012-02-19 21:30:00|
|69328   |2012-02-19 14:00:00|2012-02-19 21:30:00|
+--------+-------------------+-------------------+
only showing top 15 rows


In [24]:
# Define time intervals
during_match = (
    (col("time") >= col("kickoff_time")) &
    (col("time") <= col("kickoff_time") + expr("INTERVAL 2 HOURS"))
)

rain_window = (
    (col("time") >= col("kickoff_time") - expr("INTERVAL 2 HOURS")) &
    (col("time") <= col("kickoff_time") + expr("INTERVAL 2 HOURS"))
)

# Aggregate weather observations per match
silver_weather = weather_with_kickoff.filter(rain_window).groupBy("match_id").agg(
    avg(when(during_match, col("temp"))).alias("avg_temp"),
    avg(when(during_match, col("rhum"))).alias("avg_rel_humidity"),
    avg(when(during_match, col("wspd"))).alias("avg_wind_speed"),
    spark_sum("prcp").alias("total_precipitation"), # nedbør (regn, snø osv)
    count(when(during_match, lit(1))).alias("weather_observations"),
    count("prcp").alias("precipitation_observations")
)

silver_weather.show(10, truncate=False)

+--------+------------------+------------------+------------------+-------------------+--------------------+--------------------------+
|match_id|avg_temp          |avg_rel_humidity  |avg_wind_speed    |total_precipitation|weather_observations|precipitation_observations|
+--------+------------------+------------------+------------------+-------------------+--------------------+--------------------------+
|69329   |16.099999999999998|74.66666666666667 |13.0              |NULL               |3                   |0                         |
|69324   |17.066666666666666|61.333333333333336|15.5              |NULL               |3                   |0                         |
|265896  |20.599999999999998|95.33333333333333 |5.666666666666667 |NULL               |3                   |0                         |
|267499  |19.03333333333333 |61.333333333333336|34.1              |NULL               |3                   |0                         |
|266871  |5.0               |83.0              |

### Data Quality After Cleaning

In [25]:
print("Number of weather records (after cleaning):", silver_weather.count())
print("Number of unique match IDs (after cleaning):", silver_weather.select("match_id").distinct().count())
print("Duplicate match IDs (after cleaning):", silver_weather.groupBy("match_id").count().filter(col("count") > 1).count())
print("Invalid weather observations (after cleaning):", valid_weather.filter(invalid_weather).count())
print("Quarantined weather observations:", quarantine_weather.count())

# Missing values after aggregation
for column in ["avg_temp", "avg_rel_humidity", "avg_wind_speed", "total_precipitation"]:
    missing = silver_weather.filter(col(column).isNull()).count()
    print(f"Missing values in {column}: {missing}")

Number of weather records (after cleaning): 491
Number of unique match IDs (after cleaning): 491
Duplicate match IDs (after cleaning): 0
Invalid weather observations (after cleaning): 0
Quarantined weather observations: 0
Missing values in avg_temp: 13
Missing values in avg_rel_humidity: 14
Missing values in avg_wind_speed: 14
Missing values in total_precipitation: 487


### Join Weather With Matches

In [26]:
# Join matches with aggregated weather data
silver_matches_weather = silver_matches.join(
    silver_weather,
    on="match_id",
    how="left"
)

print("Number of matches before join:", silver_matches.count())
print("Number of matches after join:", silver_matches_weather.count())

silver_matches_weather.show(5, truncate=False)

Number of matches before join: 867
Number of matches after join: 867
+--------+------------+--------------------------+----------------------+--------------+------------+------------+----------+----------+---------+----------+--------+-----------------+-----------------+-------------------+--------------------+--------------------------+
|match_id|kick_off    |stadium                   |home_team             |away_team     |home_team_id|away_team_id|home_score|away_score|season   |match_date|avg_temp|avg_rel_humidity |avg_wind_speed   |total_precipitation|weather_observations|precipitation_observations|
+--------+------------+--------------------------+----------------------+--------------+------------+------------+----------+----------+---------+----------+--------+-----------------+-----------------+-------------------+--------------------+--------------------------+
|3825848 |20:00:00.000|Estadio Ciudad de Valencia|Levante UD            |Eibar         |221         |322         |2   

## 5. Write Silver Data To Parquet

In [27]:
# Save Silver datasets
silver_matches_weather.write.mode("overwrite").parquet("data/silver/matches")

silver_events.write.mode("overwrite").parquet("data/silver/events")

silver_weather.write.mode("overwrite").parquet("data/silver/weather")

# Save quarantined records
quarantine_matches.write.mode("overwrite").parquet("data/silver/quarantine/matches")

quarantine_events.write.mode("overwrite").parquet("data/silver/quarantine/events")

quarantine_weather.write.mode("overwrite").parquet("data/silver/quarantine/weather")

26/11/27 20:22:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
